# Graph Attention Networks (GAT)

## Learning Objectives

1. **Explain** why fixed aggregation weights are limiting in GNNs
2. **Derive** the attention mechanism for graph neighborhoods
3. **Implement** a single GAT layer with multi-head attention
4. **Compare** GAT to GCN and GraphSAGE aggregation

## Motivation: Weighted Neighbor Aggregation

In **GCN**, neighbor aggregation uses fixed structural weights (inverse degree):
$$h_v^{(l+1)} = \sigma\left(\sum_{u \in N(v) \cup \{v\}} \frac{1}{\sqrt{d_u d_v}} W h_u^{(l)}\right)$$

These weights are purely structural — determined by the graph topology.

**Limitation:** different neighbors may be more/less relevant depending on the task.
A paper's neighbor (another paper) that cites the same topic should contribute more.

**Graph Attention Networks (Veličković et al. 2018):** learn attention weights from node features:
$$\alpha_{uv} \propto \exp(\text{LeakyReLU}(a^\top [W h_u \| W h_v]))$$

## GAT Attention Mechanism

For each edge $(u, v)$, compute attention coefficient:

$$e_{uv} = \text{LeakyReLU}(\mathbf{a}^\top [\mathbf{W} h_u \| \mathbf{W} h_v])$$

Normalize over neighbors via softmax:
$$\alpha_{uv} = \frac{\exp(e_{uv})}{\sum_{k \in N(v)} \exp(e_{kv})}$$

Aggregate:
$$h_v^{\text{new}} = \sigma\left(\sum_{u \in N(v)} \alpha_{uv} \mathbf{W} h_u\right)$$

**Multi-head attention:** run $K$ independent attention heads; concatenate or average the results.

In [ ]:
import math, random

def dot(a, b): return sum(x*y for x,y in zip(a,b))

def leaky_relu(x, alpha=0.2):
    return x if x >= 0 else alpha * x

def softmax(vals):
    m = max(vals); exps = [math.exp(v-m) for v in vals]
    s = sum(exps); return [e/s for e in exps]

class GATLayer:
    def __init__(self, in_dim, out_dim, n_heads=1, seed=0):
        self.in_dim = in_dim; self.out_dim = out_dim; self.n_heads = n_heads
        rng = random.Random(seed)
        scale = 0.1
        self.W = [[[rng.gauss(0,scale) for _ in range(in_dim)] for _ in range(out_dim)]
                  for _ in range(n_heads)]
        # Attention vector a has size 2*out_dim
        self.a = [[rng.gauss(0,scale) for _ in range(2*out_dim)] for _ in range(n_heads)]

    def transform(self, h, head):
        # W h_v: (out_dim,)
        return [dot(self.W[head][i], h) for i in range(self.out_dim)]

    def attention_coef(self, Whu, Whv, head):
        concat = Whu + Whv
        return leaky_relu(dot(self.a[head], concat))

    def forward(self, H, adj):
        """H: list of node feature vectors. adj: dict of sets."""
        n = len(H)
        out_all = []
        for h_idx in range(self.n_heads):
            # Transform all nodes
            WH = [self.transform(H[v], h_idx) for v in range(n)]
            # Compute attention for each node
            new_H = []
            for v in range(n):
                neighbors = list(adj[v]) + [v]  # include self
                evals = [self.attention_coef(WH[u], WH[v], h_idx) for u in neighbors]
                alphas = softmax(evals)
                agg = [sum(alphas[k] * WH[neighbors[k]][d] for k in range(len(neighbors)))
                       for d in range(self.out_dim)]
                new_H.append(agg)
            out_all.append(new_H)
        # Concatenate heads
        return [[x for head_out in [out_all[h][v] for h in range(self.n_heads)] for x in head_out]
                for v in range(n)]

from collections import defaultdict
# Small graph
edges = [(0,1),(1,2),(2,3),(3,4),(4,0),(1,3)]
n = 5
adj = defaultdict(set)
for u,v in edges: adj[u].add(v); adj[v].add(u)

rng = random.Random(1)
H = [[rng.gauss(0,1) for _ in range(4)] for _ in range(n)]
gat = GATLayer(in_dim=4, out_dim=3, n_heads=2, seed=42)
H_new = gat.forward(H, adj)
print(f"Input  dim: {len(H[0])}")
print(f"Output dim: {len(H_new[0])} (2 heads × 3 = 6)")
print(f"Node 0 new embedding: {[round(x,4) for x in H_new[0]]}")

In [ ]:
# Examine attention weights to see what each node attends to
def get_attention(gat, H, adj, query_node, head=0):
    n = len(H)
    WH = [gat.transform(H[v], head) for v in range(n)]
    v = query_node
    neighbors = list(adj[v]) + [v]
    evals = [gat.attention_coef(WH[u], WH[v], head) for u in neighbors]
    alphas = softmax(evals)
    return list(zip(neighbors, alphas))

print("Attention weights from Node 0's perspective (head 0):")
attns = get_attention(gat, H, adj, query_node=0, head=0)
for nbr, alpha in sorted(attns, key=lambda x:-x[1]):
    tag = "(self)" if nbr == 0 else ""
    bar = '#'*int(alpha*30)
    print(f"  Node {nbr} {tag}: α={alpha:.4f}  {bar}")

## Comparison: GCN vs GraphSAGE vs GAT

| Method | Weights | Parameters | Expressiveness |
|--------|---------|-----------|----------------|
| **GCN** | Fixed: $1/\sqrt{d_u d_v}$ | $W$ only | Low (structural only) |
| **GraphSAGE** | Fixed uniform (or LSTM) | $W$ + aggregator | Medium |
| **GAT** | Learned per-edge attention | $W + a$ per head | High |
| **GAT multi-head** | $K$ independent attention heads | $K(W + a)$ | Higher (stable) |

**GAT advantage:** attention allows the model to learn *which* neighbors matter for each task.

**GAT cost:** $O(|E|)$ attention computations per layer; more parameters than GCN.

**When to use GAT:** heterogeneous neighborhoods where some neighbors are much more relevant than others (citation networks, social graphs, knowledge graphs).